[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shashvindu/ai-teacher-notebooks/blob/main/NLP/nlp_03_tokenization.ipynb) - run this notebook in your browser, nothing to install.

# NLP #3: Tokenization (Word, Sentence, Subword)

**AI Teacher** · Interview prep notebook for the video *Tokenization in NLP* (taught by Gonu Jha Ji with Priya, Rahul and Sneha).

Run the cells from top to bottom (`Shift + Enter`). Every output you see was produced by running this notebook. Change the examples and run them again - that is the best way to learn.

## What you will learn
1. What a token is, and why every model (including ChatGPT) works on token IDs
2. Sentence, word, character and subword tokenization - with code
3. Why `split()` is not a tokenizer
4. The OOV (out-of-vocabulary) problem, shown in code
5. **BPE (Byte Pair Encoding) implemented from scratch**
6. The real BERT WordPiece tokenizer

In [1]:
# One-time setup (uncomment if a library is missing):
# !pip install nltk scikit-learn pandas
import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
print("Ready!")

Ready!


## 1. Key definitions

| Term | Meaning |
|---|---|
| **Token** | One unit of text: a sentence, word, character or subword |
| **Tokenization** | Splitting text into tokens |
| **Token ID** | The integer the model actually sees for each token |
| **Vocabulary** | All tokens the tokenizer knows, each with an ID |
| **OOV** (out of vocabulary) | A word not in the vocabulary - word-level models turn it into `[UNK]` |
| **Subword** | A piece of a word: *token* + *##ization* |
| **BPE / WordPiece / SentencePiece** | Subword algorithms used by GPT / BERT / T5 |

> **Gonu Jha's story:** the scholar's giant unknown word was broken into small pieces Gonu already knew. *Unknown word, known pieces* = subword tokenization.

## 2. Sentence and word tokenization

In [2]:
from nltk.tokenize import sent_tokenize, word_tokenize
text = "Dr. Sharma isn't here. He'll come at 5 p.m. today!"
print("sent_tokenize:", sent_tokenize(text))
print("split()      :", text.split())
print("word_tokenize:", word_tokenize(text))

sent_tokenize: ["Dr. Sharma isn't here.", "He'll come at 5 p.m. today!"]
split()      : ['Dr.', 'Sharma', "isn't", 'here.', "He'll", 'come', 'at', '5', 'p.m.', 'today!']
word_tokenize: ['Dr.', 'Sharma', 'is', "n't", 'here', '.', 'He', "'ll", 'come', 'at', '5', 'p.m.', 'today', '!']


Notice:
- `sent_tokenize` knows that the dot in **Dr.** and **p.m.** does not end a sentence.
- `split()` glues punctuation to words: `here.` and `today!` would become separate vocabulary entries from `here` and `today`.
- `word_tokenize` splits contractions: `isn't` → `is` + `n't`, so the negation becomes its own token.

## 3. Character tokenization
Tiny vocabulary and no OOV - but sequences become very long.

In [3]:
word = "tokenization"
chars = list(word)
print(chars)
print(f"1 word -> {len(chars)} character tokens")

['t', 'o', 'k', 'e', 'n', 'i', 'z', 'a', 't', 'i', 'o', 'n']
1 word -> 12 character tokens


## 4. The OOV problem
Build a word vocabulary from training text, then tokenize a new sentence:

In [4]:
train_text = "the food was good . the delivery was fast . the movie was good"
vocab = {"[UNK]": 0}
for w in train_text.split():
    vocab.setdefault(w, len(vocab))
print("vocabulary:", vocab)

def encode(sentence):
    return [vocab.get(w, vocab["[UNK]"]) for w in sentence.lower().split()]

new = "the swiggy delivery was superfast"
print(new.split())
print(encode(new), "  <- 'swiggy' and 'superfast' became [UNK] (ID 0): their meaning is lost")

vocabulary: {'[UNK]': 0, 'the': 1, 'food': 2, 'was': 3, 'good': 4, '.': 5, 'delivery': 6, 'fast': 7, 'movie': 8}
['the', 'swiggy', 'delivery', 'was', 'superfast']
[1, 0, 6, 3, 0]   <- 'swiggy' and 'superfast' became [UNK] (ID 0): their meaning is lost


## 5. Subword tokenization: BPE from scratch
**BPE (Byte Pair Encoding)** learns its vocabulary from data:
1. Start with characters
2. Count every pair of adjacent symbols
3. Merge the most frequent pair into a new symbol
4. Add it to the vocabulary, repeat N times

Frequent words end up as one token; rare words are built from known pieces - so there is no OOV.

In [5]:
from collections import Counter

corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}      # word -> frequency
words = {tuple(w) + ("</w>",): f for w, f in corpus.items()}   # </w> marks the end of a word

def pair_counts(words):
    counts = Counter()
    for symbols, freq in words.items():
        for a, b in zip(symbols, symbols[1:]):
            counts[a, b] += freq
    return counts

def merge(words, pair):
    out = {}
    for symbols, freq in words.items():
        s, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                s.append(symbols[i] + symbols[i + 1]); i += 2
            else:
                s.append(symbols[i]); i += 1
        out[tuple(s)] = freq
    return out

for step in range(1, 7):
    counts = pair_counts(words)
    best, n = counts.most_common(1)[0]
    words = merge(words, best)
    print(f"merge {step}: {best[0]!r} + {best[1]!r} -> {best[0] + best[1]!r}  (seen {n} times)")
print()
for symbols in words:
    print(" ".join(symbols))

merge 1: 'e' + 's' -> 'es'  (seen 9 times)
merge 2: 'es' + 't' -> 'est'  (seen 9 times)
merge 3: 'est' + '</w>' -> 'est</w>'  (seen 9 times)
merge 4: 'l' + 'o' -> 'lo'  (seen 7 times)
merge 5: 'lo' + 'w' -> 'low'  (seen 7 times)
merge 6: 'n' + 'e' -> 'ne'  (seen 6 times)

low </w>
low e r </w>
ne w est</w>
w i d est</w>


`es` + `t` → `est` is learned because *newest* and *widest* share it. A brand-new word like *lowest* can now be built from `low` + `est` - no OOV.

## 6. The real BERT tokenizer (WordPiece)
BERT uses **WordPiece**: continuation pieces start with `##`.
(Needs `pip install transformers`; the tokenizer files download once.)

In [6]:
import os, warnings
os.environ["TRANSFORMERS_VERBOSITY"] = "error"   # hide "PyTorch not found": the tokenizer does not need it
warnings.filterwarnings("ignore")
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
for s in ["Tokenization is fun", "unhappiness", "Swiggy delivered my biryani"]:
    pieces = tok.tokenize(s)
    print(f"{s!r:32} -> {pieces}")
print()
print("IDs the model sees:", tok("Tokenization is fun")["input_ids"])
print("[CLS] =", tok.cls_token_id, " [SEP] =", tok.sep_token_id)

'Tokenization is fun'            -> ['token', '##ization', 'is', 'fun']
'unhappiness'                    -> ['un', '##ha', '##pp', '##iness']
'Swiggy delivered my biryani'    -> ['sw', '##ig', '##gy', 'delivered', 'my', 'bi', '##rya', '##ni']

IDs the model sees: [101, 19204, 3989, 2003, 4569, 102]
[CLS] = 101  [SEP] = 102


## 7. Word vs character vs subword

| | Word | Character | Subword (BPE / WordPiece) |
|---|---|---|---|
| Vocabulary size | Huge (100k+) | Tiny (~100) | Medium (30k-100k) |
| Sequence length | Short | Very long | Short |
| OOV problem | Yes → `[UNK]` | No | Almost never |
| Used by | Classical NLP | Some spelling models | BERT, GPT, T5, every modern LLM |

## 8. Interview questions (from the video)

**Q1. What is tokenization and why is it needed?**
Breaking text into tokens (words, subwords or characters). Models need fixed units mapped to IDs, then to
vectors. Tokenization decides vocabulary size, sequence length and OOV handling - and for LLMs it also
decides **cost and context-window usage** (APIs bill per token).

**Q2. Word vs character vs subword tokenization?**
Word: meaningful but huge vocabulary + OOV. Character: no OOV but very long sequences. Subword: the balance -
that is why every modern transformer uses it.

**Q3. BPE vs WordPiece?**
BPE merges the most *frequent* pair (GPT, RoBERTa). WordPiece merges the pair that most increases the
*likelihood* of the data: score = freq(ab) / (freq(a) × freq(b)) (BERT). WordPiece marks continuations with
`##`; GPT's byte-level BPE works on bytes, so it can encode any text.

**Q4. What is the OOV problem and how is it solved?**
A word not in the training vocabulary. Word-level models map it to `[UNK]` and lose its meaning. Subword
tokenizers split it into known pieces (`sw ##ig ##gy`); byte-level BPE can encode any text, so OOV disappears.

### Common mistakes
- Saying "tokenization is just `split()`"
- Assuming 1 token = 1 word (English: about 0.75 words per token)
- Using a different tokenizer than the one the model was trained with
- Forgetting that Hindi and other Indian languages often need more tokens for the same sentence

## 9. Practice
**Task (from the video):** how many BERT tokens is *"ChatGPT is awesome"*? Guess first, then run:

In [7]:
# Solution
pieces = tok.tokenize("ChatGPT is awesome")
print(pieces, "->", len(pieces), "tokens (plus [CLS] and [SEP] when fed to the model)")

['chat', '##gp', '##t', 'is', 'awesome'] -> 5 tokens (plus [CLS] and [SEP] when fed to the model)


**Next video:** NLP #4 - Stemming vs Lemmatization.